# Практика 13. Точкові оцінки та перевірка гіпотез: обчислення p-value, інтерпретація результату

**Студент:** Гузь Юрій  
**Група:** ІТ-41  
**Варіант:** 3, Одеса  
**Липнева температура для генерації:** 23 °C  
**Стара норма середньої липневої температури (μ₀):** 21.8 °C

Мета: порівняти вибірку денних температур із заявленою старою нормою, обчислити z-статистику та p-value, прийняти рішення при α = 0.05 і перевірити його через довірчий інтервал.

## Завдання 1. Вибірка і точкові оцінки

Для Одеси генеруємо одну вибірку з 30 денних температур навколо липневого параметра генерації 23 °C. Її середнє й стандартне відхилення є точковими оцінками.

In [ ]:
import numpy as np
from scipy import stats

variant = 3
july_temperature = 23
mu0 = 21.8
alpha = 0.05

np.random.seed(variant)
daily_temps = np.random.normal(loc=july_temperature, scale=2.5, size=30)

x_bar = daily_temps.mean()
s = daily_temps.std(ddof=1)
n = len(daily_temps)

print("Вибірка денних температур (°C):")
print(daily_temps)
print(f"n = {n}, x̄ = {x_bar:.3f} °C, s = {s:.3f} °C")

## Завдання 2. Формулювання H₀ і H₁

Питання сформульоване як перевірка, чи стара норма більше не відповідає дійсності, без попереднього припущення про напрям зміни. Тому до перегляду p-value обираємо двобічний тест:

- **H₀:** μ = 21.8 °C — стара норма Одеси узгоджується з істинною середньою липневою температурою.
- **H₁:** μ ≠ 21.8 °C — істинна середня липнева температура відрізняється від старої норми.

Двобічний тест враховує відхилення як у бік потепління, так і в бік похолодання. Тип тесту не змінюватимемо після перегляду результатів.

## Завдання 3. z-статистика і p-value

Обчислимо стандартну похибку за вибірковим стандартним відхиленням, z-статистику та двобічне p-value.

In [ ]:
se = s / np.sqrt(n)
z = (x_bar - mu0) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print(f"Стара норма μ₀ = {mu0:.2f} °C")
print(f"Стандартна похибка SE = {se:.4f} °C")
print(f"z = {z:.4f}")
print(f"Двобічне p-value = {p_value:.6f}")

p-value — це ймовірність, за умови істинності H₀, отримати таку саму або більш екстремальну за модулем z-статистику. Це не ймовірність того, що H₀ правильна: тест обчислює ймовірність даних за припущення H₀, а не ймовірність гіпотези за даними.

## Завдання 4. Рішення при α = 0.05 і висновок

Відхиляємо H₀, якщо p-value менше за α; в іншому разі не відхиляємо її. «Не відхилити» не означає довести, що стара норма правильна.

In [ ]:
reject_h0 = p_value < alpha
decision = "відхиляємо H₀" if reject_h0 else "не відхиляємо H₀"

print(f"α = {alpha:.2f}; p-value = {p_value:.6f} → {decision}.")
if reject_h0:
    conclusion = (
        f"Для Одеси вибірка дає статистично значущі підстави вважати, що середня липнева "
        f"температура відрізняється від старої норми {mu0:.1f} °C "
        f"(x̄ = {x_bar:.2f} °C, двобічний тест, p = {p_value:.4f}, α = {alpha:.2f})."
    )
else:
    conclusion = (
        f"Для Одеси вибірка не дає достатніх статистичних підстав стверджувати, що середня "
        f"липнева температура відрізняється від старої норми {mu0:.1f} °C "
        f"(x̄ = {x_bar:.2f} °C, двобічний тест, p = {p_value:.4f}, α = {alpha:.2f}); "
        "це не є доказом точної рівності середньої температури старій нормі."
    )
print(conclusion)

## Завдання 5. Перевірка через довірчий інтервал і помилки I/II роду

Побудуємо 95% інтервал за вказаною в завданні z-апроксимацією `x̄ ± 1.96·SE` і перевіримо, чи входить до нього μ₀. Для двобічного z-тесту при α = 0.05 рішення узгоджується з 95% z-довірчим інтервалом: H₀ відхиляється, коли μ₀ поза інтервалом.

In [ ]:
ci_95 = (x_bar - 1.96 * se, x_bar + 1.96 * se)
mu0_in_ci = ci_95[0] <= mu0 <= ci_95[1]
ci_decision_matches_test = (not mu0_in_ci) == reject_h0

print(f"95% довірчий інтервал для середньої температури: ({ci_95[0]:.3f}, {ci_95[1]:.3f}) °C")
print(f"Чи потрапляє μ₀ = {mu0:.1f} °C в інтервал? {mu0_in_ci}")
print(f"Висновок інтервального методу узгоджується з тестом? {ci_decision_matches_test}")

Для Одеси **помилка I роду** означала б відхилити H₀ і заявити, що середня липнева температура відрізняється від 21.8 °C, хоча стара норма насправді правильна. **Помилка II роду** означала б не відхилити H₀, хоча істинна середня температура вже змінилася. Для кліматологів, які готують офіційний довідник, помилка II роду може бути дорожчою, якщо через неї довідник не оновлять і рішення надалі спиратимуться на застарілу норму; водночас помилкова зміна офіційної норми через помилку I роду теж має наслідки. Відносна ціна залежить від призначення довідника та наслідків рішень на його основі.

## Контрольні питання

1. **Що означає p-value?** За умови істинності H₀ це ймовірність отримати результат не менш екстремальний, ніж спостережений. Це не ймовірність того, що стара норма правильна, бо p-value обчислюється за припущення H₀, а не вимірює ймовірність гіпотези.

2. **Чому тип тесту обирають наперед?** Одно- чи двобічний тест визначається дослідницьким питанням до перегляду даних. Вибір напрямку після результату штучно змінював би критерій екстремальності та збільшував ризик помилкового відхилення H₀.

3. **Що означають помилки I та II роду в цій задачі?** Помилка I роду — заявити, що норма Одеси 21.8 °C змінилася, хоча вона правильна. Помилка II роду — не виявити зміну середньої липневої температури, хоча вона відбулася. Це різні ситуації: перша є хибним сигналом про зміну, друга — пропущеною реальною зміною.

4. **Чи узгоджується висновок тесту з довірчим інтервалом?** За двобічного тесту й відповідного довірчого інтервалу рівня `1-α` вони дають те саме рішення: H₀ відхиляється тоді й лише тоді, коли μ₀ не входить до інтервалу. У цій роботі порівнюються саме z-тест і 95% z-інтервал з однаковою стандартною похибкою.

## Підсумок

Для вибірки денних температур Одеси перевірено двобічну гіпотезу про стару норму μ₀ = 21.8 °C при α = 0.05. Розраховані z-статистика й p-value використані для рішення, яке звірено з 95% довірчим інтервалом; також сформульовано зміст помилок I та II роду.